# HinglishSense v3 — train E3 (XLM-R) and E4 (HingRoBERTa) on Kaggle

**Before running:** Settings → Accelerator → **GPU P100** (or GPU T4 x2) · Settings → **Internet on** · Add Input → your private **hinglishsense-kaggle-bundle** dataset.

Then **Save Version → Save & Run All (Commit)** so it runs in the background (≈2–3 h). When it finishes, open the version → **Output** → download `hinglishsense_v3_models.zip`.

What it does (identical for E3 and E4): learning-rate search {1e-5, 2e-5, 3e-5} on the validation split → best rate re-run with seeds 42/43/44 → served model = best validation seed. Full fp32 AdamW, up to 30 epochs, early stopping (patience 3) on validation Macro-F1. **The test split is never used here.**

In [ ]:
import glob, os, shutil, subprocess
cands = sorted(glob.glob('/kaggle/input/**/configs/kaggle_v3.yaml', recursive=True))
assert cands, 'Attach the HinglishSense bundle dataset via "Add Input".'
src = os.path.dirname(os.path.dirname(cands[0]))
ROOT = '/kaggle/working/hs'
if not os.path.exists(ROOT):
    shutil.copytree(src, ROOT)
print('project root:', ROOT, '<-', src)
# Same library versions as the local serving environment (models must load there).
subprocess.run('pip install -q "transformers==4.46.3" "tokenizers==0.20.3"', shell=True, check=True)
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)

In [ ]:
# One GPU only: with "T4 x2" the Trainer would otherwise use both GPUs and silently double the batch size.
os.environ.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'HINGLISH_PROJECT_ROOT': ROOT,
    'PYTHONPATH': ROOT + '/python/src',
    'HINGLISH_CONFIG_OVERLAY': 'configs/kaggle_v3.yaml',
    'TOKENIZERS_PARALLELISM': 'false',
})
p = subprocess.Popen('python -u -m hinglish.kaggle_v3 --out /kaggle/working/export', shell=True, cwd=ROOT,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
assert p.wait() == 0, 'training failed - see the log above and /kaggle/working/logs/'

In [ ]:
import json
s = json.load(open('/kaggle/working/export/v3_search_summary.json'))
for e in s['experiments']:
    print(e['experiment_id'], 'chosen lr', e['chosen_learning_rate'], '| lr search', e['learning_rate_search'],
          '| seeds', e['confirm_seeds'], '| mean %.4f sd %.4f' % (e['val_macro_f1_mean'], e['val_macro_f1_sd']))
# Keep the output small: only the export zip, the summary and logs are kept.
shutil.rmtree(ROOT + '/artifacts/models', ignore_errors=True)
shutil.rmtree('/kaggle/working/export/models', ignore_errors=True)
shutil.copy('/kaggle/working/export/v3_search_summary.json', '/kaggle/working/v3_search_summary.json')
print(os.path.getsize('/kaggle/working/hinglishsense_v3_models.zip') / 2**30, 'GB ->', 'hinglishsense_v3_models.zip')